## Crash severity

The fixed rules are in `src/data/severity.py`. See [severity rules](../data/README.md#crash-severity-rules)
for precedence, missing outcomes, and the training class counts.

We review labels using 2020-2022 only, then apply the same rules to every year.
`unknown` stays in the saved files. Exclude it from model training and evaluation
and report how many rows were removed.

Outcome fields and label reasons must not be predictors. IDs and survey-design
fields are not predictors either; use `WEIGHT` only for sample weighting.

The classifier also returns `INJURY_ONLY_SEVERITY` for the training comparison.
Both labels are outcomes, never predictors. We predict severity categories using
classification models. Review [predictor availability](../data/README.md#predictor-availability)
before claiming prediction before the outcome is known.


## 1. Set up paths and shared functions
Run the cells in order from `notebooks/`. Each step below uses the same functions
as `build_data.py`. Intermediate tables remain available for inspection.

In [ ]:
from pathlib import Path
import sys

if Path.cwd().name != "notebooks":
    raise RuntimeError("Run this notebook from notebooks/.")
sys.path.insert(0, str(Path.cwd().parent))
from src.data.preprocess import (
    notebook_root, input_hashes, load_tables, prepare_crashes,
    split_crashes, processed_schema, save_datasets,
)
from src.data.joins import audit_raw_joins
from src.data.severity import classify_crashes, training_distribution

PROJECT_ROOT = notebook_root()

## Missing values and codes

The shared reader preserves valid text such as `WRK_ZONENAME = "None"`.
Only empty CSV cells are read as null. We do not fill work-zone labels or replace
8/9/98/99 globally: those numbers mean different things in different fields.

See [missing values and coding consistency](../data/README.md#missing-values-and-coding-consistency)
for the reports and handling rules. Regenerate them from the repository root
with `python -m src.data.audit_codes`. Learn imputation and encoding from training
only; validation and test checks must not change those choices.


## 2. Load the raw tables
Record input hashes before loading. The loader checks year-specific codes and
adds `YEAR_FILE`. These counts describe table structure, not holdout outcomes.

In [ ]:
raw_hashes = input_hashes(PROJECT_ROOT)
tables = load_tables(PROJECT_ROOT)
accident = tables["accident"]
vehicle = tables["vehicle"]
person = tables["person"]
for name, frame in tables.items():
    print(name, frame.shape)

## 3. Check keys and join coverage
Check duplicates, missing keys, and parent matches before grouping records.
Crashes without person records remain visible in this audit.

In [ ]:
join_audit = audit_raw_joins(accident, vehicle, person)
join_audit

## 4. Assign severity
Apply the fixed rules to every crash. Display label counts and reasons for
2020-2022 only; do not use holdout outcomes to choose rules.

In [ ]:
labels = classify_crashes(accident, vehicle, person)
training_labels = labels.loc[labels.YEAR_FILE.isin([2020, 2021, 2022])]
severity_report = training_distribution(training_labels)
severity_report

## 5. Prepare the crash-level table
Attach the primary severity label and remove the fixed excluded fields.
The schema shows column roles. Retained fields still need predictor review.

In [ ]:
crashes = prepare_crashes(accident, labels)
schema = processed_schema(crashes)
print(crashes.shape)
schema

## 6. Create and check chronological splits
The shared split function checks years, keys, columns, labels, and complete
crash coverage. Inspect training rows here; reserve holdout outcomes for evaluation.

In [ ]:
splits = split_crashes(crashes)
train = splits["train"]
validation = splits["validation"]
test = splits["test"]
print(train.shape, validation.shape, test.shape)
train.head()

## 7. Save and verify
This cell replaces the local processed files. It rechecks the splits and raw
hashes, reloads temporary CSVs to verify their contents, and writes the manifest,
schema, integrity report, join report, and training-only severity reports.
If you change raw files, restart from the loading step before saving.

In [ ]:
save_datasets(PROJECT_ROOT, splits, crashes, labels, join_audit, raw_hashes)
print("Saved and verified train.csv, validation.csv, and test.csv.")

## Output and Handoff

### Files saved
Saved to `data/processed/` (local only, not committed to GitHub):

| File | Years | Rows | Purpose |
|---|---|---|---|
| `train.csv` | 2020-2022 | 162,900 | Model training and fitting preprocessing |
| `validation.csv` | 2023 | 50,103 | Hyperparameter tuning and model selection |
| `test.csv` | 2024 | 51,658 | Final evaluation only |

Each file has 71 columns, one row per crash, with the target in the `SEVERITY` column (`high`, `moderate`, `low`, `unknown`).

### Class counts
See [the training report](../docs/crss_severity_training.csv) for 2020-2022
counts and label reasons. Holdout outcomes are not used to choose label rules.

### Still to do (not done in this notebook)
- Filter `unknown` rows before training and evaluation, and report how many were removed.
- Choose features (core: roadway, environment, vehicle; expanded: plus driver-related). The `*NAME` columns are text copies of codes and can be dropped.
- Treat code columns as categories and handle missing codes by field using the documented rules.
- Fit all imputing, encoding and scaling on `train` only.
- Do not use `test.csv` for any decisions. Use `WEIGHT` only as a sample weight.
The [processed schema](../docs/crss_processed_schema.csv) lists all 71 columns,
storage types, sources, and roles. Retained fields are not an approved predictor list.
See [rebuilding the datasets](../data/README.md#rebuilding-the-datasets) for the
command, integrity checks, and local manifest.
